In [1]:
import os
file = os.listdir('/workspaces/Edu_Math_tutor/ITF-narciss/logs')

In [ ]:
from google import genai
from google.genai import types
import json
from openai import OpenAI
import numpy as np
import os
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=,
)

prompt = f'''System Prompt:
You are an impartial evaluator.
Your task is to analyze a full dialogue between a Socratic tutor and a simulated student, together with a specification of the student behavior and the exercise requirements.
Your goal is to evaluate the quality and coherence of the tutoring interaction using seven objective rubrics.
You must base all judgments strictly on:
The conversation content:
The student behavior configuration: the student is highly engaged and ready to follow tutor instruction
The tutor goal configuration:
Guide the student to reflect fully on their own reasoning about an error they have already identified, ensuring they:
Recognize the error.
Understand why it occurred.
Can reason about how to correct it.
Do not: teach, explain, solve, or extend beyond this purpose.
Judgement requirements
Do not infer emotions, intentions, or hidden states not present in the text.
Do not reward verbosity or politeness.
Focus entirely on structural, logical, and pedagogical correctness.
EVALUATION RUBRICS (7 TOTAL)
For each rubric, you must output:
score: number between 0 and 1
explanation: 1–3 sentences
1. Tutoring Flow Consistency (0–1)
Evaluate whether the tutor maintains a coherent Socratic process aligned with the task goal.
A consistent flow avoids logical drift, maintains a clear purpose, and follows a reasonable progression (elicitation → diagnosis → guidance → consolidation), without arbitrarily jumping between goals.
Scoring guide:
1.0 = Fully coherent and aligned with the learning goal
0.5 = Some drift or inconsistent steps
0.0 = Frequent drift or contradictory guidance
2. Instruction-Following Accuracy of the Student (0–1)
Evaluate whether the student responses follow the tutor’s instructions, given the student behavior configuration (e.g., weakness patterns, error persistence, limited ability).
Scoring guide:
1.0 = Student consistently follows instructions within their constraints
0.5 = Mixed following and ignoring instructions
0.0 = Frequent or systematic ignoring
3. Turn Count to Full Task Completion
Count how many student–tutor turns occur until the student meets all task requirements.
If the student never finishes, write "incomplete".
4. Turn Count Until Correct Reasoning Direction
Identify the turn when the student’s reasoning becomes correct or aligned with the intended conceptual approach.
If the student never reaches correct reasoning direction, write "never".
5. Quality of Tutor Socratic Questioning (0–1)
Evaluate whether tutor questions are effective, clear, and pedagogically valid:
encourages reasoning (not giving answers)
non-leading
logically connected
specific to the student’s current mistake
Scoring guide:
1.0 = Questions are precise, well-scaffolded, and cognitively productive
0.5 = Some leading or vague questions
0.0 = Poor or unhelpful questioning
6. Alignment of Tutor Feedback With Student Errors (0–1)
Evaluate whether tutor feedback correctly diagnoses the student’s actual errors.
Scoring guide:
1.0 = Feedback consistently matches real mistakes
0.5 = Mixed accuracy
0.0 = Frequent misdiagnosis or irrelevant feedback
7. Student Knowledge Recall Accuracy (0–1)
Evaluate correctness of any knowledge the student recalls (formulas, definitions, reasoning steps).
Scoring guide:
1.0 = Mostly correct recall
0.5 = Mixed correctness
0.0 = Mostly incorrect or fabricated recall
OUTPUT FORMAT (JSON)
You must output only a JSON object in the following structure:
{{
  "tutoring_flow_consistency": {{
    "score": <0-1>,
    "explanation": "<short explanation>"
}},
  "instruction_following": {{
    "score": <0-1>,
    "explanation": "<short explanation>"
}},
  "turns_to_completion": "<number or 'incomplete'>",
  "turns_to_correct_direction": "<number or 'never'>",
  "socratic_question_quality": {{
    "score": <0-1>,
    "explanation": "<short explanation>"
}},
  "feedback_alignment": {{
    "score": <0-1>,
    "explanation": "<short explanation>"
  }},
  "knowledge_recall_accuracy": {{
    "score": <0-1>,
    "explanation": "<short explanation>"
  }}
}}
Do not include any extra commentary before or after the JSON.
'''
output_file = "responses_trial_gpt.jsonl"

# Prepare extra columns in df_gt
#df_gt["task_type"] = None
#df_gt["pred_wrong_solution"] = None
for f_name in file:
    dir = '/workspaces/Edu_Math_tutor/ITF-narciss/logs'
    f_path = os.path.join(dir,f_name)
    with open(f_path, "r", encoding="utf-8") as f:
      text = f.read()
    #sample = df_gt.iloc[idx]
    #content = json.loads(text)
    #+ sample['wrong_solution']

    '''response = client.models.generate_content(
        model='models/gemini-flash-latest',
        config=types.GenerateContentConfig(
            system_instruction=prompt,
            temperature=1,
            thinking_config=types.ThinkingConfig(thinking_budget=-1)
            #response_mime_type='application/json',
        ),
        contents=text,
    )'''

    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b:novita",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": text},
        ],
        temperature=0
    )
    response = completion.choices[0].message.content
    print(response)
    '''# Parse the JSON returned by Gemini
    try:
        response_json = json.loads(response.text)
    except json.JSONDecodeError as e:
        print(f"JSON decode error at idx {idx}: {e}")
        continue
    print(response_json)
    try:
        if isinstance(response_json, dict):
          data = response_json  # already a dict
        else:
          data = json.loads(response_json)

        # ensure it's a dict
        if not isinstance(data, dict):
            raise ValueError("JSON is not a dict")
        # safe extraction
        df_gt.at[idx, "pred_error_type"]     = data.get("error_type", np.nan)
        df_gt.at[idx, "pred_explanation"]    = data.get("explanation", np.nan)

    except Exception as e:
        # ANY error → set NaN for this sample
        print(f"idx {idx}: extraction failed → {e}")
        df_gt.at[idx, "pred_error_type"]     = np.nan
        df_gt.at[idx, "pred_explanation"]    = np.nan'''

    # ---- Save JSONL file for debugging ----
    with open(output_file, "a", encoding="utf-8") as f:
        json.dump(response, f, ensure_ascii=False)
        f.write("\n")
    #df_gt.at[idx, "task_type"] = response.text
    #print(f"Saved + merged response for idx {idx}")
    #break
    #if idx == 0:
        #break

{
  "tutoring_flow_consistency": {
    "score": 1.0,
    "explanation": "The tutor maintains a steady Socratic progression, repeatedly probing the student's misunderstanding without deviating from the goal."
  },
  "instruction_following": {
    "score": 1.0,
    "explanation": "The student consistently answers each of the tutor's prompts, reflecting the configured high engagement."
  },
  "turns_to_completion": "incomplete",
  "turns_to_correct_direction": "never",
  "socratic_question_quality": {
    "score": 1.0,
    "explanation": "Questions are clear, non‑leading, and directly target the student's mistaken reasoning about the number of solutions."
  },
  "feedback_alignment": {
    "score": 1.0,
    "explanation": "The tutor’s probing aligns with the student's error of stopping after the first solution and guides toward recognizing the missing solution."
  },
  "knowledge_recall_accuracy": {
    "score": 1.0,
    "explanation": "The student correctly computes the y‑coordinates for

KeyboardInterrupt: 